# 06 — Classical Model Optimization

Purpose: Build on Notebook 05's tuned baselines with advanced optimization techniques: feature selection (RFE/SelectKBest to reduce ~194 dims), Bayesian hyperparameter tuning (Optuna for top models like SVM/XGBoost), and ensemble methods (Voting/Staking for improved robustness to imbalance). Evaluate optimized models on test set with detailed metrics (ROC-AUC, PR curves for grinder recall), save best ensemble as final classical baseline. Aligns with Training Plan Section 4.3 (optimization) and Training Code Section 4.4 (Optuna/RFE/Staking).

Requires: classical_features.csv from 03/05, tuned models from 05. Outputs: optimized_features.csv, best_ensemble.pkl, optimization_results.csv, ROC/PR plots.

In [1]:
# Imports and setup
import os
from pathlib import Path
import yaml
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_selection import RFE, SelectKBest, f_classif
from sklearn.ensemble import StackingClassifier, VotingClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, precision_recall_curve, average_precision_score,
)
from sklearn.preprocessing import label_binarize
import joblib
import optuna
from optuna.integration import OptunaSearchCV
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from sklearn.metrics import roc_curve, auc

plt.style.use("seaborn-v0_8")
sns.set_context("notebook")

# Project setup (same as 05)
PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
CFG_PATH = PROJECT_ROOT / 'config.yaml'
FEATURES_PATH = PROJECT_ROOT / 'data' / 'processed' / 'classical_features' / 'classical_features.csv'
MODELS_DIR = PROJECT_ROOT / 'models' / 'classical'
METRICS_DIR = PROJECT_ROOT / 'results' / 'metrics'
FIG_DIR = PROJECT_ROOT / 'results' / 'figures'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Load config
with open(CFG_PATH, 'r') as f:
    cfg = yaml.safe_load(f)

train_cfg = cfg.get('training', {})
RANDOM_STATE = int(train_cfg.get('random_state', 42))
TEST_SIZE = float(train_cfg.get('test_size', 0.2))
VAL_SIZE = float(train_cfg.get('val_size', 0.2))
np.random.seed(RANDOM_STATE)

print('Project root:', PROJECT_ROOT)
print('Features:', FEATURES_PATH)
print('Random state:', RANDOM_STATE)


Project root: /Users/harryirving/Development/projects/ai-ml/BikeAIv4
Features: /Users/harryirving/Development/projects/ai-ml/BikeAIv4/data/processed/classical_features/classical_features.csv
Random state: 42


In [2]:
# Load features and splits from 05 (or recreate)
df_features = pd.read_csv(FEATURES_PATH)
print('Loaded features:', df_features.shape)

# Identify label and path columns dynamically
label_col = 'label'
path_cols = [c for c in df_features.columns if 'path' in c.lower()]  # e.g., 'path', 'segment_path', 'original_path'

# Drop non-feature columns
cols_to_drop = [label_col] + path_cols
X = df_features.drop(columns=[c for c in cols_to_drop if c in df_features.columns]).values
y_str = df_features[label_col].values

# Load label encoder from correct path (Notebook 05 saves to models/classical/)
le_path = PROJECT_ROOT / 'models' / 'classical' / 'label_encoder.pkl'
if not le_path.exists():
    # Fallback to root models/ if old structure
    le_path = PROJECT_ROOT / 'models' / 'label_encoder.pkl'
    if not le_path.exists():
        raise FileNotFoundError(f'Label encoder not found. Run Notebook 05 first.')
le = joblib.load(le_path)
y = le.transform(y_str)
classes = le.classes_

print('Features after drop:', X.shape)
print('Classes:', classes)

# Stratified split (80/20 test, then split train into 80/20 for train/val → ~60/20/20 overall)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
# Val size relative to temp (which is 80% of original)
val_fraction = VAL_SIZE / (1 - TEST_SIZE)  # e.g., 0.2 / 0.8 = 0.25 for 20% val
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=val_fraction, random_state=RANDOM_STATE, stratify=y_temp
)

print(f'\nSplits: Train {X_train.shape[0]}, Val {X_val.shape[0]}, Test {X_test.shape[0]}')
print(f'Class dist train: {np.bincount(y_train, minlength=len(classes))}')
print(f'Class dist val:   {np.bincount(y_val, minlength=len(classes))}')
print(f'Class dist test:  {np.bincount(y_test, minlength=len(classes))}')


Loaded features: (19044, 366)
Features after drop: (19044, 364)
Classes: ['angle_grinder' 'background' 'tools']

Splits: Train 13330, Val 1905, Test 3809
Class dist train: [4464 5648 3218]
Class dist val:   [638 807 460]
Class dist test:  [1276 1614  919]


In [3]:
# Feature Selection: Reduce dims from ~194 to 50-100 (RFE + univariate)
# RFE with tuned SVM from 05 (recursive backward elimination)
svm_tuned = joblib.load(PROJECT_ROOT / 'models' / 'classical' / 'svm_tuned.pkl')  # Load from 05
n_features_rfe = int(train_cfg.get('n_features_rfe', 100))
rfe = RFE(estimator=svm_tuned, n_features_to_select=n_features_rfe, step=0.1)
rfe.fit(X_train, y_train)
X_train_rfe = rfe.transform(X_train)
X_val_rfe = rfe.transform(X_val)
X_test_rfe = rfe.transform(X_test)

# Univariate SelectKBest (ANOVA F-value for multi-class)
k_best = int(train_cfg.get('n_features_kbest', 50))
select_kbest = SelectKBest(score_func=f_classif, k=k_best)
select_kbest.fit(X_train_rfe, y_train)
X_train_sel = select_kbest.transform(X_train_rfe)
X_val_sel = select_kbest.transform(X_val_rfe)
X_test_sel = select_kbest.transform(X_test_rfe)

# Save selected features
feature_mask = rfe.support_ & select_kbest.get_support()
selected_names = [df_features.columns[i] for i in range(len(feature_mask)) if feature_mask[i]]
pd.DataFrame({'feature': selected_names}).to_csv(METRICS_DIR / 'selected_features.csv', index=False)
print(f'Selected {len(selected_names)} features (RFE+SelectKBest): {selected_names[:5]}...')
print(f'Shapes: Train {X_train_sel.shape}, Val {X_val_sel.shape}, Test {X_test_sel.shape}')


ValueError: when `importance_getter=='auto'`, the underlying estimator SVC should have `coef_` or `feature_importances_` attribute. Either pass a fitted estimator to feature selector or call fit before calling transform.

In [ ]:
# Advanced Tuning with Optuna (Bayesian for top models: SVM, XGBoost)
# Objective: Maximize F1-macro (better for imbalance than accuracy)
from sklearn.metrics import make_scorer, f1_score
f1_scorer = make_scorer(f1_score, average='macro')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# SVM Optuna (on selected features)
def svm_objective(trial):
    C = trial.suggest_float('C', 0.1, 100, log=True)
    gamma = trial.suggest_categorical('gamma', ['scale', 0.001, 0.01, 0.1, 1])
    kernel = trial.suggest_categorical('kernel', ['rbf', 'linear'])
    model = SVC(C=C, gamma=gamma, kernel=kernel, class_weight='balanced', random_state=RANDOM_STATE)
    scores = cross_val_score(model, X_train_sel, y_train, cv=cv, scoring=f1_scorer)
    return scores.mean()

study_svm = optuna.create_study(direction='maximize')
study_svm.optimize(svm_objective, n_trials=int(train_cfg.get('optuna_trials', 50)))
best_svm_params = study_svm.best_params
best_svm = SVC(**best_svm_params, class_weight='balanced', random_state=RANDOM_STATE)
best_svm.fit(X_train_sel, y_train)
joblib.dump(best_svm, MODELS_DIR / 'svm_optimized.pkl')

# XGBoost Optuna (handles imbalance well)
def xgb_objective(trial):
    params = {
        'objective': 'multi:softprob',
        'num_class': len(classes),
        'eval_metric': 'mlogloss',
        'random_state': RANDOM_STATE,
        'n_jobs': -1,
        'scale_pos_weight': {i: sum(y_train == i) / max(sum(y_train == c) for c in range(len(classes))) for i in range(len(classes))},  # Imbalance
    }
    params.update({
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
    })
    model = xgb.XGBClassifier(**params)
    scores = cross_val_score(model, X_train_sel, y_train, cv=cv, scoring=f1_scorer)
    return scores.mean()

study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(xgb_objective, n_trials=int(train_cfg.get('optuna_trials', 50)))
best_xgb_params = study_xgb.best_params
best_xgb = xgb.XGBClassifier(**best_xgb_params, random_state=RANDOM_STATE, scale_pos_weight={i: sum(y_train == i) / max(sum(y_train == c) for c in range(len(classes))) for i in range(len(classes))})
best_xgb.fit(X_train_sel, y_train)
joblib.dump(best_xgb, MODELS_DIR / 'xgb_optimized.pkl')

print(f'Best SVM F1: {study_svm.best_value:.4f}, Params: {best_svm_params}')
print(f'Best XGB F1: {study_xgb.best_value:.4f}, Params: {best_xgb_params}')


In [ ]:
# Ensembles: Voting and Stacking with top tuned models from 05 + optimized
# Load tuned from 05
rf_tuned = joblib.load(PROJECT_ROOT / 'models' / 'classical' / 'rf_tuned.pkl')
knn_tuned = joblib.load(PROJECT_ROOT / 'models' / 'classical' / 'knn_tuned.pkl')
if 'xgb_tuned.pkl' in os.listdir(PROJECT_ROOT / 'models' / 'classical'):
    xgb_tuned = joblib.load(PROJECT_ROOT / 'models' / 'classical' / 'xgb_tuned.pkl')
else:
    xgb_tuned = best_xgb  # Fallback

# Voting (hard/soft average)
voting_hard = VotingClassifier(
    estimators=[('svm', best_svm), ('rf', rf_tuned), ('knn', knn_tuned), ('xgb', xgb_tuned)],
    voting='hard'  # Majority vote
)
voting_soft = VotingClassifier(
    estimators=[('svm', best_svm), ('rf', rf_tuned), ('knn', knn_tuned), ('xgb', xgb_tuned)],
    voting='soft'  # Probability average
)
# Stacking (meta-learner on base preds)
stacking = StackingClassifier(
    estimators=[('svm', best_svm), ('rf', rf_tuned), ('knn', knn_tuned), ('xgb', xgb_tuned)],
    final_estimator=xgb.XGBClassifier(random_state=RANDOM_STATE),
    cv=3,  # Fast meta-CV
    stack_method='auto'
)

# Fit ensembles on train
for name, model in [('voting_hard', voting_hard), ('voting_soft', voting_soft), ('stacking', stacking)]:
    model.fit(X_train_sel, y_train)
    joblib.dump(model, MODELS_DIR / f'{name}.pkl')
    print(f'Fit {name}')

In [ ]:
# Evaluation: Test set metrics, plots for optimized/ensembles
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
scaler = StandardScaler()
X_test_scaled = scaler.fit_transform(X_test_sel)
X_val_scaled = scaler.transform(X_val_sel)

results = []
models_to_eval = [
    ('SVM-Opt', best_svm),
    ('XGB-Opt', best_xgb),
    ('Voting-Hard', voting_hard),
    ('Voting-Soft', voting_soft),
    ('Stacking', stacking),
]

for name, model in models_to_eval:
    y_pred = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled) if hasattr(model, 'predict_proba') else None
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')
    report = classification_report(y_test, y_pred, target_names=classes, output_dict=True)
    
    # ROC-AUC multi-class (one-vs-rest)
    auc_macro = 0
    if y_pred_proba is not None:
        y_test_bin = label_binarize(y_test, classes=range(len(classes)))
        for i in range(len(classes)):
            auc_macro += roc_auc_score(y_test_bin[:, i], y_pred_proba[:, i]) / len(classes)
    
    # PR-AUC for imbalance (grinder focus)
    pr_auc = {cls: average_precision_score(y_test == le.transform([cls]), y_pred_proba[:, le.transform([cls])[0]] if y_pred_proba is not None else 0) for cls in classes}
    
    results.append({
        'model': name,
        'accuracy': acc,
        'f1_macro': f1,
        'roc_auc_macro': auc_macro,
        'pr_auc_grinder': pr_auc.get('angle_grinder', 0),  # Key metric
        'report': report,
    })
    
    print(f'\n{name}: Acc {acc:.4f}, F1 {f1:.4f}, ROC {auc_macro:.4f}, PR-Grinder {pr_auc.get("angle_grinder", 0):.4f}')
    print(classification_report(y_test, y_pred, target_names=classes))
    
    # Confusion Matrix plot
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.title(f'CM: {name}')
    plt.ylabel('True')
    plt.xlabel('Pred')
    plt.savefig(FIG_DIR / f'cm_{name.lower().replace(" ", "_")}.png', dpi=150)
    plt.close()
    
    # ROC/PR curves if proba available
    if y_pred_proba is not None:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
        
        # ROC
        colors = ['blue', 'red', 'green']
        for i, cls in enumerate(classes):
            fpr, tpr, _ = roc_curve((y_test == i).astype(int), y_pred_proba[:, i])
            roc_auc = auc(fpr, tpr)
            ax1.plot(fpr, tpr, color=colors[i], lw=2, label=f'{cls} (AUC = {roc_auc:.2f})')
        ax1.plot([0, 1], [0, 1], 'k--', lw=1)
        ax1.set_xlim([0.0, 1.0])
        ax1.set_ylim([0.0, 1.05])
        ax1.set_xlabel('FPR')
        ax1.set_ylabel('TPR')
        ax1.set_title(f'ROC Curves: {name}')
        ax1.legend(loc='lower right')
        
        # PR (focus on grinder)
        grinder_idx = le.transform(['angle_grinder'])[0]
        precision, recall, _ = precision_recall_curve(y_test == grinder_idx, y_pred_proba[:, grinder_idx])
        pr_auc = average_precision_score(y_test == grinder_idx, y_pred_proba[:, grinder_idx])
        ax2.plot(recall, precision, color='orange', lw=2, label=f'Grinder (AP = {pr_auc:.2f})')
        ax2.set_xlabel('Recall')
        ax2.set_ylabel('Precision')
        ax2.set_title(f'PR Curve: {name}')
        ax2.legend(loc='lower left')
        plt.tight_layout()
        plt.savefig(FIG_DIR / f'roc_pr_{name.lower().replace(" ", "_")}.png', dpi=150)
        plt.close()



In [ ]:
# Save results
df_results = pd.DataFrame(results)
df_results.to_csv(METRICS_DIR / 'classical_optimization_results.csv', index=False)

# Best ensemble (e.g., stacking if best)
best_model = stacking  # Or select by F1/ROC
joblib.dump(best_model, MODELS_DIR / 'best_classical_ensemble.pkl')
joblib.dump((scaler, select_kbest, rfe), MODELS_DIR / 'feature_pipeline.pkl')  # Save pipeline

# Comparison table (vs 05 baselines)
if Path(METRICS_DIR / 'tuned_results.csv').exists():
    df_tuned = pd.read_csv(METRICS_DIR / 'tuned_results.csv')
    comparison = pd.concat([df_tuned[['model', 'val_accuracy']].rename(columns={'val_accuracy': 'Baseline Val'}), 
                            df_results[['model', 'accuracy']].rename(columns={'accuracy': 'Optimized Test'})], axis=1)
    print(comparison)
    comparison.to_csv(METRICS_DIR / 'baseline_vs_optimized.csv', index=False)
else:
    print('No baseline CSV found—run 05 first')

print('Optimization complete. Best model saved for deployment baselines.')


In [ ]:
%run 07_custom_cnn_training.ipynb